# 05 — Conversation fine-tuning of shona-mt5-small (Kaggle)

Phase 3 of TauraBot. Takes the Phase 2 model `mathiaskabango/shona-mt5-small` and fine-tunes it on hand-crafted Shona Q&A pairs to produce the conversational chatbot model `mathiaskabango/taurabot-shona`.

---
## Setup (do once on Kaggle's website)

1. **Update the `taurabot-code` Kaggle Dataset** with the latest repo (now containing `src/data/conversation.py`, `src/model/finetune.py`, `configs/finetune.yaml`). Use Dataset → *New Version* with the updated `taurabot-code.tar.gz`.
2. **Create a new Kaggle Dataset `taurabot-conversations`** containing just `conversation_pairs.json` (the file you author locally). Mark **private**. You'll re-upload as new versions when you add more pairs.
3. **`HF_TOKEN` secret** — already configured from Phase 2; verify it's toggled on for this notebook.
4. **Attach datasets** (`taurabot-code` + `taurabot-conversations`) and enable **GPU T4×2**.

Fine-tuning is much cheaper than pretraining — for 500–1000 pairs at 3 epochs, expect **15–30 minutes** of GPU time.

In [ ]:
# Adjust paths to match your Kaggle dataset slugs
CODE_DATASET_PATH          = '/kaggle/input/taurabot-code'
CONVERSATIONS_DATASET_PATH = '/kaggle/input/taurabot-conversations'

WORK = '/kaggle/working'
OUTPUT_DIR = f'{WORK}/checkpoints/taurabot-shona'

BASE_MODEL    = 'mathiaskabango/shona-mt5-small'    # Phase 2 model
HUB_MODEL_ID  = 'mathiaskabango/taurabot-shona'      # Phase 3 output target

## 1. Verify GPU + inputs

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os
for p in (CODE_DATASET_PATH, CONVERSATIONS_DATASET_PATH):
    print(f"{'OK' if os.path.exists(p) else 'MISSING':>8}  {p}")
!ls $CODE_DATASET_PATH 2>/dev/null | head -10
!ls -lh $CONVERSATIONS_DATASET_PATH 2>/dev/null

## 2. Stage code + conversation pairs

In [ ]:
import os, shutil, glob

PROJECT_DIR = f'{WORK}/taurabot'
if not os.path.exists(PROJECT_DIR):
    shutil.copytree(CODE_DATASET_PATH, PROJECT_DIR)

# Find the conversation_pairs.json inside the conversations dataset
matches = glob.glob(f'{CONVERSATIONS_DATASET_PATH}/**/conversation_pairs.json', recursive=True)
assert matches, f'conversation_pairs.json not found inside {CONVERSATIONS_DATASET_PATH}'
pairs_local = f'{PROJECT_DIR}/conversation_pairs.json'
if os.path.exists(pairs_local):
    os.remove(pairs_local)
os.symlink(matches[0], pairs_local)

print('staged code →', PROJECT_DIR)
print('pairs file  →', pairs_local)
!ls -l $pairs_local

## 3. Quick validation of the conversation pairs

Catches schema errors + reports topic balance + flags `TEMPLATE_REPLACE_ME` placeholders **before** burning GPU time.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)
from src.data.conversation import load_conversation_pairs, summarize_pairs

pairs = load_conversation_pairs(pairs_local)
summary = summarize_pairs(pairs)
print('Total pairs:           ', summary['total_pairs'])
print('By topic:              ', summary['by_topic'])
print('Missing planned topics:', summary['missing_topics'])
print('Unknown topics used:   ', summary['unknown_topics'])
print('Avg context words:     ', summary['avg_context_words'])
print('Avg response words:    ', summary['avg_response_words'])

templates_left = [p.id for p in pairs if 'TEMPLATE_REPLACE_ME' in p.id]
if templates_left:
    print(f"\n⚠️  {len(templates_left)} template placeholders still unfilled: {templates_left[:5]}{'...' if len(templates_left)>5 else ''}")
    print('   Replace them with real Shona pairs before training, or you will fine-tune the model on placeholder text.')
if summary['total_pairs'] < 50:
    print(f"\n⚠️  Only {summary['total_pairs']} pairs — fine-tuning will probably overfit.")
    print('   Project plan targets 500–1000 pairs; consider authoring more before running.')

## 4. Install training dependencies (same as Phase 2)

In [ ]:
!pip install -q -U 'transformers>=4.40,<5.0' 'accelerate>=1.0,<2.0' \
                   'datasets>=2.18,<3.0' 'huggingface_hub>=0.24,<1.0' \
                   'sentencepiece>=0.1.99' 'protobuf>=3.20,<5.0' \
                   'evaluate>=0.4' pyyaml tensorboard
import torch; print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'n_gpus:', torch.cuda.device_count())

## 5. HuggingFace Hub login

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
print('HF Hub login OK.')

## 6. Run fine-tuning

Saves best checkpoint by `eval_loss` (small dataset = real overfit risk; we don't want the final epoch by default). Auto-pushes the best model to `mathiaskabango/taurabot-shona`.

Expected runtime for 500–1000 pairs × 3 epochs: ~15–30 minutes on T4×2.

In [ ]:
import subprocess, os

cmd = [
    'python', '-m', 'src.model.finetune',
    '--config', 'configs/finetune.yaml',
    '--pairs_path', pairs_local,
    '--output_dir', OUTPUT_DIR,
    '--base_model', BASE_MODEL,
    '--push_to_hub', 'true',
    '--hub_model_id', HUB_MODEL_ID,
]
print('Running:', ' '.join(cmd))
os.chdir(PROJECT_DIR)
subprocess.run(cmd, check=True)

## 7. Sanity-check the chatbot

Generates responses to a few held-out prompts. Quality is your call — these go in front of you, the fluent Shona speaker.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
mdl = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to('cuda').eval()

prompts = [
    'Mhoro! Makadini?',
    'Zita rangu ndiTaura. Munobva kupi?',
    'Ndinokukumbira kuti unditaurire zita rako.',
    'Ndingakubatsira sei nhasi?',
    'Chikafu chaunoda chii?',
]
for p in prompts:
    inputs = tok(f'Mubvunzo: {p}\nMhinduro:', return_tensors='pt', truncation=True).to('cuda')
    out = mdl.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.3,
    )
    print(f'\nIN:  {p}')
    print(f'OUT: {tok.decode(out[0], skip_special_tokens=True)}')

## What's next

If the responses look reasonable for the topics you covered, Phase 3 is done. The model is at `mathiaskabango/taurabot-shona` on the Hub, ready for Phase 4 — the Gradio chatbot deployment.

If responses are off-topic, repetitive, or grammatically broken:
- **Off-topic:** more pairs in the missing topics (look at the validation cell's `missing_topics` list)
- **Repetitive:** lower `repetition_penalty` to 1.1, or add more diverse pairs per topic
- **Broken grammar:** add more pairs OR increase `num_train_epochs` from 3 to 5 (small risk of overfit)

Iterate: edit `conversation_pairs.json`, re-upload as a new version of the `taurabot-conversations` dataset, refresh the notebook input, re-run.